In [15]:
!pip install -q chroma openai
!pip install -q chromadb openai pandas

In [16]:
from pathlib import Path
from pprint import pprint
from pydantic import BaseModel
from typing import List

def print_response(response):
    print(f"Reponse id: {response.id}")
    print(f"Status: {response.status}")
    print(f"Input tokens: {response.usage.input_tokens} ({response.usage.input_tokens_details.cached_tokens} cached) | Output tokens: {response.usage.output_tokens} ({response.usage.output_tokens_details.reasoning_tokens} reasoning)")
    pprint(response.output)

    print()
    print(f"{'=' * 20} [Text] {'=' * 20}")
    print(response.output_text)

In [17]:
from google.colab import userdata
from openai import OpenAI

api_key = userdata.get('OPENROUTER_API_KEY')
client = OpenAI(api_key=api_key, base_url="https://openrouter.ai/api/v1")

In [18]:
#get cached info for user
from pathlib import Path

class MemorableResponse(BaseModel):
    response: str
    facts_to_remember: List[str]


cache_path = Path("/content/cache.txt")

def get_cached_data() -> List[str]:
  with open(cache_path, 'r') as f:
    list = f.readlines()
    cleaned_list = [row.replace("\n", "") for row in list]

  return cleaned_list



def save_facts(response: MemorableResponse) -> None:

  with cache_path.open(mode="a") as file:
      for fact in response.facts_to_remember:
        file.write(f"\n{fact}")


  return get_cached_data()

In [19]:


##Using AI with CAG(Cache Augmented Generation) Кеширано усилено генериране here we are passing information of the model every time on every single request and we could edit and add that information(can be file)

# This is a typical example for CAG (Cache Augmented Generation)
# We take all the known external information and provide it to the AI model in advance

def ask(question: str, history = []):
    known_facts = get_cached_data()
    answer_response = client.responses.parse(
        model="gpt-5-nano",
        input=[
            { "role": "developer", "content": "You are a helpful AI assistant that specializes in day-to-day interactions. Keep track of various interesting facts about the user that you may use in future to provide better responses and be even more helpful! If a fact is already known, there is no need to include it in the output." },
            *history,
            { "role": "assistant", "content": f"Facts I know about the user: {';'.join(known_facts)}"},
            { "role": "user", "content": question }
        ],
        text_format=MemorableResponse
    )

    save_facts(answer_response.output_parsed)

    return answer_response




In [5]:
greeting_prompt = "Hello, i am from small city called Lovech located in Bulgaria, where are you from?"
ask_about_favorite_game = "I feel fine. The weather is sunny. I just won on fifa which is my favorite game in the world! Do you have any favorite game?"
where_are_you_from_prompt = "And what about your location? I live in Bulgaria."

In [8]:
greeting_response = ask(greeting_prompt)

In [9]:
game_response = ask(ask_about_favorite_game)

In [10]:
print_response(game_response)

Reponse id: gen-1776144203-eMxrbDb4l54AiUFnHoMZ
Status: completed
Input tokens: 188 (0 cached) | Output tokens: 1344 (1216 reasoning)
[ResponseReasoningItem(id='rs_tmp_tjlr56419l', summary=[Summary(text='**Considering user engagement**\n\nI want to ask about the user’s favorite aspect of FIFA and suggest other games with a similar vibe, like eFootball, NBA 2K, or Rocket League. I should note their good mood, especially since they just won at FIFA while enjoying sunny weather. It would be great to get them to share details about the match, their favorite team, and preferred gameplay modes. Additionally, I must track facts about the user, particularly their location in Lovech, Bulgaria, and their win in FIFA.**Storing user information**\n\nI want to record that the user’s favorite game is FIFA, especially since they emphasized that it\'s "their favorite game in the world!" I need to incorporate this into the facts while ensuring my response is concise and friendly. I should format this a

In [22]:

from chromadb import PersistentClient
from pprint import pprint
from pydantic import BaseModel
from rich.console import Console
from rich.table import Table
import pandas as pd

class Book(BaseModel):
    title: str
    author: str
    description: str

books_path = Path("/content/goodreads_dataset.csv")


In [23]:
df = pd.read_csv(books_path)

# Filter books without description
df = df[df['description'].notna()]

In [24]:
books = [Book(**row) for row in df.to_dict(orient="records")]

In [25]:
pprint(books)

[Book(title="Harry Potter and the Sorcerer's Stone (Harry Potter, #1)", author='J.K. Rowling', description="Harry Potter's life is miserable. His parents are dead and he's stuck with his heartless relatives, who force him to live in a tiny closet under the stairs. But his fortune changes when he receives a letter that tells him the truth about himself: he's a wizard. A mysterious visitor rescues him from his relatives and takes him to his new home, Hogwarts School of Witchcraft Harry Potter's life is miserable. His parents are dead and he's stuck with his heartless relatives, who force him to live in a tiny closet under the stairs. But his fortune changes when he receives a letter that tells him the truth about himself: he's a wizard. A mysterious visitor rescues him from his relatives and takes him to his new home, Hogwarts School of Witchcraft and Wizardry.After a lifetime of bottling up his magical powers, Harry finally feels like a normal kid. But even within the Wizarding communit

In [26]:
max_description_length = max(len(b.description) for b in books)
print(f"Max description length: {max_description_length}")

Max description length: 2783


In [27]:
PATH_TO_CHROMA_DB = "/content/chromadb"
chroma_client = PersistentClient(path=PATH_TO_CHROMA_DB)

In [28]:
chroma_collection = chroma_client.get_or_create_collection(name="goodreads-experiments")

In [29]:

# Identify books by one-based index
ids = [f"{i + 1}" for i in range(len(books))]
metadatas = [{ "title": b.title, "author": b.author } for b in books]
documents = [b.description for b in books]

chroma_collection.upsert(
    ids=ids,
    documents=documents,
    metadatas=metadatas
)

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:06<00:00, 13.3MiB/s]


In [30]:
print(ids)
print(metadatas)
print(documents)

['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87']
[{'title': "Harry Potter and the Sorcerer's Stone (Harry Potter, #1)", 'author': 'J.K. Rowling'}, {'title': 'The Book Thief', 'author': 'Markus Zusak'}, {'title': 'The Hunger Games (The Hunger Games, #1)', 'author': 'Suzanne Collins'}, {'title': 'Harry Potter and the Goblet of Fire (Harry Potter, #4)', 'author': 'J.K. Rowling'}, {'title': "The Handmaid's Tale (The Handmaid's Tale, #1)", 'author': 'Margaret Atwood'}, {'title': 'Harry Potter and the Deathly Hallows (Harry Potter, #7)

In [31]:
results = chroma_collection.get(include=["documents", "metadatas", "embeddings"])

pd.DataFrame({
    "id": results["ids"],
    "document": results["documents"],
    "metadata": results["metadatas"],
    "embeddings": results["embeddings"].tolist()
})

,id,document,metadata,embeddings
0,1,Harry Potter's life is miserable. His parents ...,"{'author': 'J.K. Rowling', 'title': 'Harry Pot...","[-0.02532176487147808, 0.0678311362862587, 0.0..."
1,2,Librarian's note: An alternate cover edition c...,"{'title': 'The Book Thief', 'author': 'Markus ...","[-0.06795577704906464, 0.021274710074067116, -..."
2,3,"Could you survive on your own in the wild, wit...","{'author': 'Suzanne Collins', 'title': 'The Hu...","[0.06891398876905441, 0.031230002641677856, 0...."
3,4,Harry Potter is midway through his training as...,"{'author': 'J.K. Rowling', 'title': 'Harry Pot...","[0.026427583768963814, 0.003385880496352911, 0..."
4,5,Offred is a Handmaid in the Republic of Gilead...,"{'author': 'Margaret Atwood', 'title': 'The Ha...","[-0.023029835894703865, -0.08016649633646011, ..."
...,...,...,...,...
82,83,"In 1920s London, Virginia Woolf is fighting ag...","{'author': 'Michael Cunningham', 'title': 'The...","[-0.010132635943591595, -0.07977994531393051, ..."
83,84,Pulitzer Prize Winner (1998)In American Pastor...,"{'author': 'Philip Roth', 'title': 'American P...","[0.019527697935700417, 0.0048889899626374245, ..."
84,85,"In nineteenth-century China, in a remote Hunan...","{'title': 'Snow Flower and the Secret Fan', 'a...","[-0.10635953396558762, 0.029351208359003067, 0..."
85,86,"Days before his release from prison, Shadow's ...","{'title': 'American Gods (American Gods, #1)',...","[-0.027844272553920746, 0.0008544745505787432,..."


In [32]:
def print_query_results(result):
    console = Console()

    queries_count = len(result["ids"])
    for i in range(queries_count):
        table = Table(show_lines=True, expand=True)
        table.add_column("#")
        table.add_column("Title")
        table.add_column("Author")
        table.add_column("Description")

        results_count = len(result["ids"][i])
        for j in range(results_count):
            table.add_row(result["ids"][i][j], result["metadatas"][i][j]["title"], result["metadatas"][i][j]["author"], result["documents"][i][j])

        console.print(table)

In [57]:
query_result = chroma_collection.query(
    query_texts=[
        "Find books about magic that include young boy whos life is tough and hard, and he is super unhappy"
    ],
    n_results=1,
    include=["metadatas", "documents"]
)

In [58]:

print_query_results(query_result)

┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ # ┃ Title                                        ┃ Author       ┃ Description                                   ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1 │ Harry Potter and the Sorcerer's Stone (Harry │ J.K. Rowling │ Harry Potter's life is miserable. His parents │
│   │ Potter, #1)                                  │              │ are dead and he's stuck with his heartless    │
│   │                                              │              │ relatives, who force him to live in a tiny    │
│   │                                              │              │ closet under the stairs. But his fortune      │
│   │                                              │              │ changes when he receives a letter that tells  │
│   │                                              │              │ him the truth about himself: he's a wizard. A │
│   │                                              │              │ mysterious visitor rescues him from his       │
│   │                                              │              │ relatives and takes him to his new home,      │
│   │                                              │              │ Hogwarts School of Witchcraft Harry Potter's  │
│   │                                              │              │ life is miserable. His parents are dead and   │
│   │                                              │              │ he's stuck with his heartless relatives, who  │
│   │                                              │              │ force him to live in a tiny closet under the  │
│   │                                              │              │ stairs. But his fortune changes when he       │
│   │                                              │              │ receives a letter that tells him the truth    │
│   │                                              │              │ about himself: he's a wizard. A mysterious    │
│   │                                              │              │ visitor rescues him from his relatives and    │
│   │                                              │              │ takes him to his new home, Hogwarts School of │
│   │                                              │              │ Witchcraft and Wizardry.After a lifetime of   │
│   │                                              │              │ bottling up his magical powers, Harry finally │
│   │                                              │              │ feels like a normal kid. But even within the  │
│   │                                              │              │ Wizarding community, he is special. He is the │
│   │                                              │              │ boy who lived: the only person to have ever   │
│   │                                              │              │ survived a killing curse inflicted by the     │
│   │                                              │              │ evil Lord Voldemort, who launched a brutal    │
│   │                                              │              │ takeover of the Wizarding world, only to      │
│   │                                              │              │ vanish after failing to kill Harry.Though     │
│   │                                              │              │ Harry's first year at Hogwarts is the best of │
│   │                                              │              │ his life, not everything is perfect. There is │
│   │                                              │              │ a dangerous secret object hidden within the   │
│   │                                              │              │ castle walls, and Harry believes it's his     │
│   │                                              │              │ responsibility to prevent it from falling     │
│   │                                              │    